# ResLSTM-SER: Speech Emotion Recognition using Attention-based LSTM-Network with Residual Connection

Reproducible training notebook for the DSPA 2026 paper:
**"Speech Emotion Recognition using Attention-based LSTM-Network with Residual Connection"**
by Daniil Krasnoproshin and Maxim Vashkevich.

> IEEE Xplore: https://ieeexplore.ieee.org/document/11476771

This notebook performs:
1. Feature extraction (MFCC + chromagram) from RAVDESS .wav files
2. Global mean/std normalization
3. Model training with Optuna hyperparameter optimization
4. 10-run statistical evaluation with optimal hyperparameters
5. K-fold cross-validation (speaker-independent, 5-fold)

In [6]:
import torch
print(f"CUDA có khả dụng không?: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Tên GPU: {torch.cuda.get_device_name(0)}")

CUDA có khả dụng không?: True
Tên GPU: NVIDIA GeForce RTX 4060 Laptop GPU


## Imports & Path Configuration

In [7]:
import glob
import os
import sys
from typing import List, Tuple

import numpy as np
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
from ignite.metrics.recall import Recall
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset
from tqdm import tqdm

# ── Paths — ADJUST THESE ──
# The RAVDESS dataset is NOT included in this repository.
# Download from: https://www.kaggle.com/datasets/uwrfkaggler/ravdess-emotional-speech-audio
# or: https://zenodo.org/record/1188976

ROOT_DIR = os.path.abspath('')  # current working directory
RAVDESS_DATA_PATH = os.path.join(ROOT_DIR, 'data', 'ravdess', 'audio_speech_actors_01-24', '**', '*.wav')
EXTRACTED_DATA_DIR = os.path.join(ROOT_DIR, 'data', 'ravdess', 'extracted')
MODEL_BACKUP_DIR = os.path.join(ROOT_DIR, 'model_backup')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
print(f"RAVDESS path: {RAVDESS_DATA_PATH}")

Device: cuda
RAVDESS path: d:\Speech_Processing\ResLSTM-SER\data\ravdess\audio_speech_actors_01-24\**\*.wav


d:\Speech_Processing\ResLSTM-SER\.venv\Lib\site-packages\ignite\handlers\checkpoint.py:16: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import ZeroRedundancyOptimizer


## Reproducibility

In [8]:
def set_seed(seed: int = 42):
    """Set random seed for reproducibility."""
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

## RAVDESS Label Mappings

In [9]:
def get_emotions():
    return ["neutral", "calm", "happy", "sad", "angry", "fear", "disgust", "surprised"]

def get_emotions_dictionary():
    return {
        "01": "neutral",
        "02": "calm",
        "03": "happy",
        "04": "sad",
        "05": "angry",
        "06": "fear",
        "07": "disgust",
        "08": "surprised",
    }

def get_actors():
    return {f"{i:02d}": str(i) for i in range(1, 25)}

## Feature Extraction (wav → npy)

In [10]:
def extract_features(file_path: str,
                     n_mfcc: int = 34,
                     n_chroma: int = 12,
                     n_fft: int = 4096,
                     hop_length: int = None) -> np.ndarray:
    """
    Extract MFCC and chromagram features from an audio file.

    Args:
        file_path: Path to the audio file.
        n_mfcc: Number of MFCC features to extract.
        n_chroma: Number of chroma bins to produce.
        n_fft: FFT window size.
        hop_length: Number of samples between successive frames
                    (default: n_fft // 2).

    Returns:
        features: (time_len, n_mfcc + n_chroma) feature array, or None on error.
    """
    import librosa

    try:
        audio, sample_rate = librosa.load(file_path, sr=None)

        if hop_length is None:
            hop_length = n_fft // 2

        mfcc = librosa.feature.mfcc(
            y=audio, sr=sample_rate, n_mfcc=n_mfcc,
            n_fft=n_fft, hop_length=hop_length
        ).T

        chroma = librosa.feature.chroma_stft(
            y=audio, sr=sample_rate, n_fft=n_fft,
            n_chroma=n_chroma, hop_length=hop_length
        ).T

        features = np.concatenate([mfcc, chroma], axis=1)
        return features
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

### Two-pass Feature Extraction with Global Normalization

In [11]:
def process_dataset(dataset_path: str,
                    x_output_path_template: str,
                    y_output_path_template: str,
                    id_output_path_template: str,
                    n_mfcc: int,
                    n_chroma: int,
                    n_fft: int,
                    hop_length: int):
    """
    Two-pass feature extraction with global mean/std normalization.

    Pass 1: accumulate global statistics (mean, std) across all files.
    Pass 2: normalize each file and save as .npy.
    """
    valid_files = []
    valid_emotions = []
    valid_actors = []

    emotions_dict = get_emotions_dictionary()
    actors_dict = get_actors()
    observed_emotions = get_emotions()

    print("Pass 1: collecting global statistics...")
    total_sum = None
    total_sq_sum = None
    total_samples = 0

    all_files = list(glob.glob(dataset_path))
    if not all_files:
        raise FileNotFoundError(
            f"No audio files found at {dataset_path}. "
            f"Please download the RAVDESS dataset first."
        )

    for file in tqdm(all_files, desc="Computing statistics"):
        file_name = os.path.basename(file)
        splitted = file_name.split("-")
        emotion = emotions_dict[splitted[2]]

        if emotion not in observed_emotions:
            continue

        actor = actors_dict[splitted[6].split(".")[0]]
        features = extract_features(file_path=file, n_mfcc=n_mfcc,
                                    n_chroma=n_chroma, n_fft=n_fft,
                                    hop_length=hop_length)

        if features is None:
            continue

        valid_files.append(file)
        valid_emotions.append(emotion)
        valid_actors.append(actor)

        n_samples = features.shape[0]
        if total_sum is None:
            n_features = features.shape[1]
            total_sum = np.zeros(n_features, dtype=np.float64)
            total_sq_sum = np.zeros(n_features, dtype=np.float64)

        total_sum += np.sum(features, axis=0)
        total_sq_sum += np.sum(features ** 2, axis=0)
        total_samples += n_samples

    if total_sum is None or total_samples == 0:
        raise RuntimeError("No valid data found.")

    global_mean = total_sum / total_samples
    global_var = (total_sq_sum / total_samples) - (global_mean ** 2)
    global_std = np.sqrt(np.maximum(global_var, 1e-8))

    print(f"Global stats: {total_samples} frames, {len(global_mean)} features")
    print(f"Mean (first 5): {global_mean[:5]}")
    print(f"Std  (first 5): {global_std[:5]}")

    os.makedirs(EXTRACTED_DATA_DIR, exist_ok=True)
    np.save(os.path.join(EXTRACTED_DATA_DIR, "global_mean.npy"), global_mean)
    np.save(os.path.join(EXTRACTED_DATA_DIR, "global_std.npy"), global_std)

    print("Pass 2: normalizing and saving...")
    for i, (file, emotion, actor) in enumerate(tqdm(
            zip(valid_files, valid_emotions, valid_actors),
            desc="Normalizing",
            total=len(valid_files)
    )):
        features = extract_features(file_path=file, n_mfcc=n_mfcc,
                                    n_chroma=n_chroma, n_fft=n_fft,
                                    hop_length=hop_length)
        features_norm = (features - global_mean) / global_std

        x_path = x_output_path_template.format(i, n_mfcc, n_chroma, n_fft, hop_length)
        np.save(x_path, features_norm)

    y_path = y_output_path_template.format(n_mfcc, n_chroma, n_fft, hop_length)
    np.save(y_path, np.array(valid_emotions))

    id_path = id_output_path_template.format(n_mfcc, n_chroma, n_fft)
    np.save(id_path, np.array(valid_actors))

    print(f"Processed {len(valid_files)} files.")
    return global_mean, global_std

### Run Feature Extraction

In [12]:
N_MFCC = 34
N_CHROMA = 12
N_FFT = 4096
HOP_LENGTH = N_FFT // 2

os.makedirs(EXTRACTED_DATA_DIR, exist_ok=True)

x_template = os.path.join(EXTRACTED_DATA_DIR, "X_{}_mfcc_{}_n_chroma_{}_nfft_{}_hop_size_{}.npy")
y_template = os.path.join(EXTRACTED_DATA_DIR, "y_labels_mfcc_{}_n_chroma_{}_nfft_{}_hop_size_{}.npy")
id_template = os.path.join(EXTRACTED_DATA_DIR, "IDs_mfcc_{}_n_chroma_{}_nfft_{}.npy")

y_path = y_template.format(N_MFCC, N_CHROMA, N_FFT, HOP_LENGTH)

if not os.path.exists(y_path):
    process_dataset(
        dataset_path=RAVDESS_DATA_PATH,
        x_output_path_template=x_template,
        y_output_path_template=y_template,
        id_output_path_template=id_template,
        n_mfcc=N_MFCC,
        n_chroma=N_CHROMA,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH
    )
else:
    print(f"Features already extracted at {EXTRACTED_DATA_DIR}")

Features already extracted at d:\Speech_Processing\ResLSTM-SER\data\ravdess\extracted


## PyTorch Dataset

In [13]:
class RavdessAudioDataset(Dataset):
    """RAVDESS dataset for variable-length sequences with speaker-based folds."""

    def __init__(self,
                 mfccs_dir: str,
                 annotations_file: str,
                 actor_ids_file: str,
                 desired_labels: list = None):
        self.mfccs_dir = mfccs_dir
        self.label_mapping = {
            'neutral': 0, 'calm': 1, 'happy': 2, 'sad': 3,
            'angry': 4, 'fear': 5, 'disgust': 6, 'surprised': 7
        }

        y = np.load(annotations_file, allow_pickle=True)
        ids = np.load(actor_ids_file, allow_pickle=True)

        if desired_labels is not None:
            mask = np.isin(y, desired_labels)
            y = y[mask]
            ids = ids[mask]

        self.y = torch.tensor([self.label_mapping[label] for label in y], dtype=torch.long)
        self.X_ids = [int(a) for a in ids]

        self.feature_files = []
        for i in range(len(self.y)):
            fpath = mfccs_dir.format(i, N_MFCC, N_CHROMA, N_FFT, HOP_LENGTH)
            self.feature_files.append(fpath)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index):
        features = np.load(self.feature_files[index])
        features = torch.tensor(features, dtype=torch.float32)
        label = self.y[index]
        return features, label

    def get_kth_fold_inds(self, fold_num: int):
        """
        Speaker-independent 5-fold split (same as paper).
        Returns (train_indices, val_indices, test_indices).
        """
        folds = [
            [2, 5, 14, 15, 16],
            [3, 6, 7, 13, 18],
            [10, 11, 12, 19, 20],
            [8, 17, 21, 23, 24],
            [1, 4, 9, 22],
        ]
        folds_val = [
            [3, 6],
            [10, 11],
            [8, 17],
            [1, 14],
            [2, 5],
        ]

        ids_train, ids_val, ids_test = [], [], []
        for i in range(len(self.X_ids)):
            if self.X_ids[i] in folds[fold_num]:
                ids_test.append(i)
            elif self.X_ids[i] in folds_val[fold_num]:
                ids_val.append(i)
            else:
                ids_train.append(i)
        return ids_train, ids_val, ids_test


def pad_mfcc_sequence(batch: List[Tuple[torch.Tensor, int]]):
    """Collate function: pad sequences and return (padded, labels, lengths)."""
    sequences, labels = zip(*batch)
    lengths = torch.tensor([seq.size(0) for seq in sequences], dtype=torch.long)
    padded = pad_sequence(sequences, batch_first=True)
    labels = torch.stack(labels) if isinstance(labels[0], torch.Tensor) else torch.tensor(labels, dtype=torch.long)
    return padded, labels, lengths

### Create Dataset Instance

In [14]:
feature_dir_template = os.path.join(EXTRACTED_DATA_DIR, "X_{}_mfcc_{}_n_chroma_{}_nfft_{}_hop_size_{}.npy")
id_path = id_template.format(N_MFCC, N_CHROMA, N_FFT)

dataset = RavdessAudioDataset(
    mfccs_dir=feature_dir_template,
    annotations_file=y_path,
    actor_ids_file=id_path,
    desired_labels=get_emotions()
)
print(f"Dataset: {len(dataset)} samples, 8 classes")

Dataset: 1440 samples, 8 classes


## Models

### ResLSTM-SA (Residual LSTM with Soft Attention)

In [15]:
class ResLSTM_SA(nn.Module):
    """
    ResLSTM-SA: LSTM with residual connection and soft attention.

    Architecture (from the paper):
        Input → LSTM1 ──(+)──→ LSTM2 → Attention → FC → Output
    """

    def __init__(self,
                 input_size: int,
                 hidden_size: int,
                 num_layers: int,
                 num_classes: int,
                 dropout_p: float = 0.1,
                 device: str = 'cpu'):
        super().__init__()
        self.device = device
        self.hidden_size = hidden_size
        self.input_size = input_size
        self.num_layers = num_layers

        # LSTM1: input_size → input_size (for residual connection)
        self.lstm1 = nn.LSTM(input_size, input_size, num_layers,
                             bidirectional=False, batch_first=True)
        # LSTM2: input_size → hidden_size
        self.lstm2 = nn.LSTM(input_size, hidden_size, num_layers,
                             bidirectional=False, batch_first=True)
        # Single-vector soft attention
        self.attention_vector = nn.Parameter(torch.empty(hidden_size, 1))
        # BatchNorm
        self.bn_residual = nn.BatchNorm1d(input_size)
        self.bn = nn.BatchNorm1d(hidden_size)
        # Classifier
        self.fc = nn.Linear(hidden_size, num_classes)
        self.drop = nn.Dropout(p=dropout_p)
        self.num_classes = num_classes
        self.initialize_weights()

    def initialize_weights(self):
        for lstm, proj_size in [(self.lstm1, self.input_size),
                                (self.lstm2, self.hidden_size)]:
            for layer in range(self.num_layers):
                nn.init.xavier_normal_(getattr(lstm, f'weight_ih_l{layer}'))
                nn.init.orthogonal_(getattr(lstm, f'weight_hh_l{layer}'))
                bias_ih = getattr(lstm, f'bias_ih_l{layer}')
                bias_hh = getattr(lstm, f'bias_hh_l{layer}')
                nn.init.zeros_(bias_ih)
                nn.init.zeros_(bias_hh)
                # Forget gate bias = 1
                with torch.no_grad():
                    bias_ih[proj_size:2 * proj_size].fill_(1.0)
                    bias_hh[proj_size:2 * proj_size].fill_(1.0)
        nn.init.xavier_normal_(self.attention_vector)
        nn.init.xavier_uniform_(self.fc.weight)
        nn.init.zeros_(self.fc.bias)

    def compute_attention(self, lstm_out):
        """Soft attention over time dimension."""
        scores = torch.matmul(lstm_out, self.attention_vector).squeeze(-1)
        weights = torch.softmax(scores, dim=1)
        context = torch.sum(lstm_out * weights.unsqueeze(-1), dim=1)
        return context, weights

    def forward(self, x, lengths, return_embeddings=False): # Đổi cờ thành return_embeddings
        batch_size = x.size(0)
        x_original = x.clone()

        # LSTM1 + residual
        h0 = torch.zeros(self.num_layers, batch_size, self.input_size).to(self.device)
        c0 = torch.zeros(self.num_layers, batch_size, self.input_size).to(self.device)
        x_packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True,
                                        enforce_sorted=False)
        lstm1_out_packed, _ = self.lstm1(x_packed, (h0, c0))
        lstm1_out, _ = pad_packed_sequence(lstm1_out_packed, batch_first=True)

        if lstm1_out.size(1) != x.size(1):
            seq_len = min(lstm1_out.size(1), x.size(1))
            lstm1_out = lstm1_out[:, :seq_len, :]
            x_original = x_original[:, :seq_len, :]

        residual = lstm1_out + x_original
        # BatchNorm1d ожидает (B, F, T)
        residual_norm = self.bn_residual(residual.transpose(1, 2)).transpose(1, 2)

        # LSTM2
        h0_l2 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(self.device)
        c0_l2 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(self.device)
        residual_packed = pack_padded_sequence(residual_norm, lengths.cpu(),
                                               batch_first=True, enforce_sorted=False)
        lstm2_out_packed, _ = self.lstm2(residual_packed, (h0_l2, c0_l2))
        lstm2_out, _ = pad_packed_sequence(lstm2_out_packed, batch_first=True)

        # --- PHẦN THAY ĐỔI Ở ĐÂY ---
        feature_vector, attention_weights = self.get_feature_vector(lstm2_out)
        
        # Gọi là embeddings cho chuẩn danh pháp của Triplet Loss
        embeddings = self.bn(feature_vector) 
        
        logits = self.fc(self.drop(embeddings))

        # Nếu cờ được bật, trả về logits và embeddings
        if return_embeddings:
            return logits, embeddings
            
        return logits

### LSTM-SA Baseline (single LSTM, no residual connection)

In [16]:
import torch.nn.functional as F

class ResLSTM_SA(nn.Module):
    """
    ResLSTM-SA: LSTM with residual connection and soft attention.
    """
    def __init__(self, input_size: int, hidden_size: int, num_layers: int, num_classes: int,
                 dropout_p: float = 0.1, device: str = 'cpu'):
        super().__init__()
        self.device = device
        self.hidden_size = hidden_size
        self.input_size = input_size
        self.num_layers = num_layers

        self.lstm1 = nn.LSTM(input_size, input_size, num_layers,
                             bidirectional=False, batch_first=True)
        self.lstm2 = nn.LSTM(input_size, hidden_size, num_layers,
                             bidirectional=False, batch_first=True)
        
        self.attention_vector = nn.Parameter(torch.empty(hidden_size, 1))
        
        self.bn_residual = nn.BatchNorm1d(input_size)
        self.bn = nn.BatchNorm1d(hidden_size)
        
        self.fc = nn.Linear(hidden_size, num_classes)
        self.drop = nn.Dropout(p=dropout_p)
        self.num_classes = num_classes
        self.initialize_weights()

    def initialize_weights(self):
        for lstm, proj_size in [(self.lstm1, self.input_size),
                                (self.lstm2, self.hidden_size)]:
            for layer in range(self.num_layers):
                nn.init.xavier_normal_(getattr(lstm, f'weight_ih_l{layer}'))
                nn.init.orthogonal_(getattr(lstm, f'weight_hh_l{layer}'))
                bias_ih = getattr(lstm, f'bias_ih_l{layer}')
                bias_hh = getattr(lstm, f'bias_hh_l{layer}')
                nn.init.zeros_(bias_ih)
                nn.init.zeros_(bias_hh)
                with torch.no_grad():
                    bias_ih[proj_size:2 * proj_size].fill_(1.0)
                    bias_hh[proj_size:2 * proj_size].fill_(1.0)
        nn.init.xavier_normal_(self.attention_vector)
        nn.init.xavier_uniform_(self.fc.weight)
        nn.init.zeros_(self.fc.bias)

    # ĐÂY LÀ HÀM TÍNH ATTENTION CỦA CLASS NÀY
    def compute_attention(self, lstm_out):
        scores = torch.matmul(lstm_out, self.attention_vector).squeeze(-1)
        weights = torch.softmax(scores, dim=1)
        context = torch.sum(lstm_out * weights.unsqueeze(-1), dim=1)
        return context, weights

    def forward(self, x, lengths, return_embeddings=False, return_attention=False):
        batch_size = x.size(0)
        x_original = x.clone()

        h0 = torch.zeros(self.num_layers, batch_size, self.input_size).to(self.device)
        c0 = torch.zeros(self.num_layers, batch_size, self.input_size).to(self.device)
        x_packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        lstm1_out_packed, _ = self.lstm1(x_packed, (h0, c0))
        lstm1_out, _ = pad_packed_sequence(lstm1_out_packed, batch_first=True)

        if lstm1_out.size(1) != x.size(1):
            seq_len = min(lstm1_out.size(1), x.size(1))
            lstm1_out = lstm1_out[:, :seq_len, :]
            x_original = x_original[:, :seq_len, :]

        residual = lstm1_out + x_original
        residual_norm = self.bn_residual(residual.transpose(1, 2)).transpose(1, 2)

        h0_l2 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(self.device)
        c0_l2 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(self.device)
        residual_packed = pack_padded_sequence(residual_norm, lengths.cpu(),
                                               batch_first=True, enforce_sorted=False)
        lstm2_out_packed, _ = self.lstm2(residual_packed, (h0_l2, c0_l2))
        lstm2_out, _ = pad_packed_sequence(lstm2_out_packed, batch_first=True)

        # --- ĐÃ SỬA: Gọi đúng tên hàm compute_attention ---
        context, attention_weights = self.compute_attention(lstm2_out)
        
        # 1. Trích xuất đặc trưng và chuẩn hóa cho Triplet Loss
        embeddings = self.bn(context)
        embeddings = F.normalize(embeddings, p=2, dim=1)
        
        # 2. Phân loại
        logits = self.fc(self.drop(embeddings))

        # 3. Điều hướng đầu ra
        if return_attention:
            return logits, attention_weights
        if return_embeddings:
            return logits, embeddings
            
        return logits

## Evaluation Utilities

In [17]:
from ignite.engine import Engine

def _eval_step(engine, batch):
    return batch

def get_default_evaluator():
    return Engine(_eval_step)

def get_test_evaluator():
    return Engine(_eval_step)

## Training Loop

In [18]:
from pytorch_metric_learning import losses, miners
import numpy as np
import torch
from ignite.metrics.recall import Recall

def get_current_alpha(epoch, warmup_epochs=15, target_alpha=0.5):
    if epoch < warmup_epochs:
        return target_alpha * (epoch / warmup_epochs)
    return target_alpha

def training_loop(model, loss_fn_ce, loss_fn_triplet, miner, target_alpha, optimizer, lr_scheduler,
                  train_loader, val_loader, n_epochs, model_path):
    best_epoch = -1
    UAR_val_max = -1
    loss_train_history = np.zeros(n_epochs)
    loss_val_history = np.zeros(n_epochs)

    scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None
    default_evaluator = get_default_evaluator()
    macro_metric = Recall(average=True)
    macro_metric.attach(default_evaluator, "macro_recall")

    for epoch in range(n_epochs):
        current_alpha = get_current_alpha(epoch, target_alpha=target_alpha)
        
        model.train()
        loss_train = 0.0
        for X, labels, lengths in train_loader:
            X, labels, lengths = X.to(DEVICE), labels.to(DEVICE), lengths.to(DEVICE)
            optimizer.zero_grad()

            if scaler is not None:
                with torch.amp.autocast('cuda'):
                    logits, embeddings = model(X, lengths, return_embeddings=True)
                    loss_ce = loss_fn_ce(logits, labels)
                    
                    hard_pairs = miner(embeddings, labels)
                    loss_trip = loss_fn_triplet(embeddings, labels, hard_pairs)
                    
                    loss = loss_ce + (current_alpha * loss_trip)
                    
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                logits, embeddings = model(X, lengths, return_embeddings=True)
                loss_ce = loss_fn_ce(logits, labels)
                hard_pairs = miner(embeddings, labels)
                loss_trip = loss_fn_triplet(embeddings, labels, hard_pairs)
                loss = loss_ce + (current_alpha * loss_trip)
                loss.backward()
                optimizer.step()
                
            loss_train += loss.item()
        loss_train /= len(train_loader)

        model.eval()
        loss_val = 0.0
        val_pred_all, val_true_all = [], []
        with torch.no_grad():
            for X, labels, lengths in val_loader:
                X, labels, lengths = X.to(DEVICE), labels.to(DEVICE), lengths.to(DEVICE)
                logits = model(X, lengths)
                loss = loss_fn_ce(logits, labels)
                loss_val += loss.item()
                val_pred_all.append(torch.softmax(logits, dim=1))
                val_true_all.append(labels)

        val_pred_all = torch.cat(val_pred_all)
        val_true_all = torch.cat(val_true_all)
        loss_val /= len(val_loader)
        loss_train_history[epoch] = loss_train
        loss_val_history[epoch] = loss_val

        default_evaluator.terminate()
        state = default_evaluator.run([[val_pred_all, val_true_all]])
        UAR_val = state.metrics["macro_recall"]

        if lr_scheduler is not None:
            lr_scheduler.step()

        if UAR_val > UAR_val_max:
            UAR_val_max = UAR_val
            best_epoch = epoch
            torch.save(model.state_dict(), model_path)

        if epoch == 0 or (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch + 1}/{n_epochs} [Alpha: {current_alpha:.2f}] | "
                  f"Train loss: {loss_train:.4f} | Val loss: {loss_val:.4f} | Val UAR: {UAR_val:.4f}")

    model.load_state_dict(torch.load(model_path, weights_only=True))
    return loss_train_history, loss_val_history, best_epoch, UAR_val_max

## K-Fold Cross-Validation

In [19]:
def k_fold_cv(dataset, model_factory, loss_fn_ce, loss_fn_triplet, miner, alpha, optimizer_factory,
              lr_scheduler_factory, n_epochs, k_fold, batch_size,
              init_model_path, model_path):
    oof_preds, oof_targets = [], []

    for fold in range(k_fold):
        print(f"\n{'='*60}")
        print(f"Fold {fold + 1}/{k_fold}")
        print(f"{'='*60}")

        inds_train, inds_val, inds_test = dataset.get_kth_fold_inds(fold)
        train_set = torch.utils.data.Subset(dataset, inds_train)
        val_set = torch.utils.data.Subset(dataset, inds_val)
        test_set = torch.utils.data.Subset(dataset, inds_test)

        train_loader = torch.utils.data.DataLoader(
            train_set, batch_size=batch_size, shuffle=True, collate_fn=pad_mfcc_sequence,
            num_workers=0, pin_memory=True)
        val_loader = torch.utils.data.DataLoader(
            val_set, batch_size=batch_size, shuffle=False, collate_fn=pad_mfcc_sequence,
            num_workers=0, pin_memory=True)
        test_loader = torch.utils.data.DataLoader(
            test_set, batch_size=batch_size, shuffle=False, collate_fn=pad_mfcc_sequence,
            num_workers=0, pin_memory=True)

        set_seed(42 + fold * 100)
        model = model_factory()
        torch.save(model.state_dict(), init_model_path)
        optimizer = optimizer_factory(model)
        lr_scheduler = lr_scheduler_factory(optimizer)

        training_loop(model, loss_fn_ce, loss_fn_triplet, miner, alpha, optimizer, lr_scheduler,
                      train_loader, val_loader, n_epochs, model_path)

        model.eval()
        fold_preds, fold_targets = [], []
        with torch.no_grad():
            for X, labels, lengths in test_loader:
                X, labels, lengths = X.to(DEVICE), labels.to(DEVICE), lengths.to(DEVICE)
                logits = model(X, lengths)
                fold_preds.append(torch.softmax(logits, dim=1).cpu())
                fold_targets.append(labels.cpu())
        oof_preds.append(torch.cat(fold_preds))
        oof_targets.append(torch.cat(fold_targets))

    return oof_preds, oof_targets

## Hyperparameter Optimization with Optuna

### Configure Search Space

In [20]:
INPUT_SIZE = N_MFCC + N_CHROMA  # 46
NUM_CLASSES = 8
NUM_LAYERS = 1
N_EPOCHS = 100
K_FOLD = 5

# Choose hidden size: 32, 64, or 128
HIDDEN_SIZE = 64

# --- THAY ĐỔI Ở ĐÂY: Tạo file DB mới và tên Study mới ---
SQLITE_STORAGE_URL = f"sqlite:///optuna_h{HIDDEN_SIZE}_triplet.db"
STUDY_NAME = f"ResLSTM_SA_H{HIDDEN_SIZE}_Triplet_tuning"

print(f"Hidden size: {HIDDEN_SIZE}, Study: {STUDY_NAME}")

Hidden size: 64, Study: ResLSTM_SA_H64_Triplet_tuning


### Objective Function

In [21]:
from pytorch_metric_learning import losses, miners

def objective(trial: optuna.Trial) -> float:
    lr = trial.suggest_float("lr", 1e-5, 3e-2, log=True)
    
    # Bỏ tùy chọn 64 để tránh lỗi Out of Memory khi dùng Triplet Loss trên card 8GB
    batch_size = trial.suggest_categorical("batch_size", [4, 8, 16, 32]) 
    
    dropout_rate = trial.suggest_float("dropout_rate", 0.02, 0.5)
    weight_decay = trial.suggest_float("wd", 2e-5, 4e-2, log=True)
    t_0_loop = trial.suggest_categorical("t_0_loop", [1, 2, 3, 5, 10])

    # Bạn có thể để Optuna tự tìm luôn trọng số Alpha tốt nhất nếu muốn
    # target_alpha = trial.suggest_float("alpha", 0.1, 1.0)
    target_alpha = 0.5 

    set_seed(42)

    def model_factory():
        return ResLSTM_SA(
            input_size=INPUT_SIZE, hidden_size=HIDDEN_SIZE,
            num_layers=NUM_LAYERS, num_classes=NUM_CLASSES,
            dropout_p=dropout_rate, device=str(DEVICE)
        ).to(DEVICE)

    def optimizer_factory(model):
        return optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    def scheduler_factory(optimizer):
        return optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=t_0_loop, T_mult=1, eta_min=lr * 0.01)

    # --- KHAI BÁO CÁC HÀM MULTI-TASK LOSS Ở ĐÂY ---
    loss_fn_ce = nn.CrossEntropyLoss()
    loss_fn_triplet = losses.TripletMarginLoss(margin=0.3)
    miner = miners.MultiSimilarityMiner()
    
    os.makedirs(MODEL_BACKUP_DIR, exist_ok=True)
    init_model_path = os.path.join(MODEL_BACKUP_DIR, f"init_{STUDY_NAME}.pt")
    model_path = os.path.join(MODEL_BACKUP_DIR, f"best_{STUDY_NAME}.pt")

    # --- CẬP NHẬT LỜI GỌI HÀM K_FOLD_CV ---
    oof_preds, oof_targets = k_fold_cv(
        dataset=dataset, model_factory=model_factory, 
        loss_fn_ce=loss_fn_ce, loss_fn_triplet=loss_fn_triplet, miner=miner, alpha=target_alpha,
        optimizer_factory=optimizer_factory, lr_scheduler_factory=scheduler_factory,
        n_epochs=N_EPOCHS, k_fold=K_FOLD, batch_size=batch_size,
        init_model_path=init_model_path, model_path=model_path)

    all_preds = torch.cat([p for p in oof_preds])
    all_targets = torch.cat([t for t in oof_targets])

    evaluator = get_default_evaluator()
    macro_metric = Recall(average=True)
    macro_metric.attach(evaluator, "macro_recall")
    evaluator.terminate()
    state = evaluator.run([[all_preds, all_targets]])
    return state.metrics["macro_recall"]

### Run Optuna Study

In [ ]:
N_TRIALS = 50  # Bạn có thể giảm xuống 50 nếu muốn test nhanh

study = optuna.create_study(
    storage=SQLITE_STORAGE_URL,
    study_name=STUDY_NAME,
    direction="maximize",
    load_if_exists=True,
)

print(f"Starting Optuna optimization with {N_TRIALS} trials...")

try:
    # Hàm optimize sẽ tự động gọi hàm objective Multi-task bạn vừa sửa
    study.optimize(objective, n_trials=N_TRIALS)
except KeyboardInterrupt:
    # Bẫy lỗi: Dừng êm đẹp nếu bạn chủ động bấm Stop (Interrupt Kernel)
    print("\nQuá trình tối ưu hóa đã được người dùng dừng lại ngang chừng.")

print("\nBest hyperparameters:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")
print(f"  Best UAR: {study.best_value:.4f}")

[I 2026-07-07 13:05:20,865] Using an existing study with name 'ResLSTM_SA_H64_Triplet_tuning' instead of creating a new one.


Starting Optuna optimization with 50 trials...

Fold 1/5
Epoch 1/100 [Alpha: 0.00] | Train loss: 1.8312 | Val loss: 1.7658 | Val UAR: 0.3750
Epoch 10/100 [Alpha: 0.30] | Train loss: 0.2394 | Val loss: 1.7531 | Val UAR: 0.4062
Epoch 20/100 [Alpha: 0.50] | Train loss: 0.1106 | Val loss: 1.9191 | Val UAR: 0.4844
Epoch 30/100 [Alpha: 0.50] | Train loss: 0.1003 | Val loss: 2.3214 | Val UAR: 0.4141
Epoch 40/100 [Alpha: 0.50] | Train loss: 0.0644 | Val loss: 2.2573 | Val UAR: 0.4766
Epoch 50/100 [Alpha: 0.50] | Train loss: 0.0495 | Val loss: 2.0802 | Val UAR: 0.5234
Epoch 60/100 [Alpha: 0.50] | Train loss: 0.0567 | Val loss: 2.3551 | Val UAR: 0.4531
Epoch 70/100 [Alpha: 0.50] | Train loss: 0.0589 | Val loss: 2.2984 | Val UAR: 0.4688
Epoch 80/100 [Alpha: 0.50] | Train loss: 0.0737 | Val loss: 2.3198 | Val UAR: 0.4375
Epoch 90/100 [Alpha: 0.50] | Train loss: 0.0689 | Val loss: 2.5471 | Val UAR: 0.5156
Epoch 100/100 [Alpha: 0.50] | Train loss: 0.1049 | Val loss: 2.5817 | Val UAR: 0.4609

Fold 2/

[I 2026-07-07 13:36:10,780] Trial 24 finished with value: 0.5188802083333333 and parameters: {'lr': 0.008266426616202284, 'batch_size': 8, 'dropout_rate': 0.20707822328150208, 'wd': 6.131301563152571e-05, 't_0_loop': 10}. Best is trial 13 with value: 0.5384114583333334.



Fold 1/5
Epoch 1/100 [Alpha: 0.00] | Train loss: 1.8433 | Val loss: 1.9545 | Val UAR: 0.2266
Epoch 10/100 [Alpha: 0.30] | Train loss: 0.3876 | Val loss: 1.5166 | Val UAR: 0.4219
Epoch 20/100 [Alpha: 0.50] | Train loss: 0.1990 | Val loss: 1.9095 | Val UAR: 0.4766
Epoch 30/100 [Alpha: 0.50] | Train loss: 0.0987 | Val loss: 1.9731 | Val UAR: 0.5156
Epoch 40/100 [Alpha: 0.50] | Train loss: 0.1024 | Val loss: 2.1167 | Val UAR: 0.4688
Epoch 50/100 [Alpha: 0.50] | Train loss: 0.0924 | Val loss: 2.2654 | Val UAR: 0.4375
Epoch 60/100 [Alpha: 0.50] | Train loss: 0.0635 | Val loss: 2.0402 | Val UAR: 0.5078
Epoch 70/100 [Alpha: 0.50] | Train loss: 0.0846 | Val loss: 2.2208 | Val UAR: 0.4922
Epoch 80/100 [Alpha: 0.50] | Train loss: 0.1119 | Val loss: 2.4728 | Val UAR: 0.4922
Epoch 90/100 [Alpha: 0.50] | Train loss: 0.1101 | Val loss: 2.1424 | Val UAR: 0.5547
Epoch 100/100 [Alpha: 0.50] | Train loss: 0.0763 | Val loss: 2.3466 | Val UAR: 0.4922

Fold 2/5
Epoch 1/100 [Alpha: 0.00] | Train loss: 1.786

[I 2026-07-07 14:03:53,898] Trial 25 finished with value: 0.482421875 and parameters: {'lr': 0.006680018456976635, 'batch_size': 8, 'dropout_rate': 0.20859117744920952, 'wd': 5.353750153102449e-05, 't_0_loop': 5}. Best is trial 13 with value: 0.5384114583333334.


...


Fold 1/5
Epoch 1/100 [Alpha: 0.00] | Train loss: 1.7802 | Val loss: 1.5960 | Val UAR: 0.4062
Epoch 10/100 [Alpha: 0.30] | Train loss: 0.3049 | Val loss: 1.3259 | Val UAR: 0.5156
Epoch 20/100 [Alpha: 0.50] | Train loss: 0.3032 | Val loss: 1.5790 | Val UAR: 0.4844
Epoch 30/100 [Alpha: 0.50] | Train loss: 0.2609 | Val loss: 1.8368 | Val UAR: 0.5156
Epoch 40/100 [Alpha: 0.50] | Train loss: 0.2475 | Val loss: 1.5451 | Val UAR: 0.5625
Epoch 50/100 [Alpha: 0.50] | Train loss: 0.2660 | Val loss: 1.7443 | Val UAR: 0.5156
Epoch 60/100 [Alpha: 0.50] | Train loss: 0.2424 | Val loss: 1.8044 | Val UAR: 0.5156
Epoch 70/100 [Alpha: 0.50] | Train loss: 0.2343 | Val loss: 1.7218 | Val UAR: 0.5078
Epoch 80/100 [Alpha: 0.50] | Train loss: 0.2303 | Val loss: 1.5329 | Val UAR: 0.4844
Epoch 90/100 [Alpha: 0.50] | Train loss: 0.2480 | Val loss: 1.8220 | Val UAR: 0.5391
Epoch 100/100 [Alpha: 0.50] | Train loss: 0.2337 | Val loss: 1.7366 | Val UAR: 0.5312

Fold 2/5
Epoch 1/100 [Alpha: 0.00] | Train loss: 1.769

[I 2026-07-07 15:45:28,810] Trial 30 finished with value: 0.5032552083333333 and parameters: {'lr': 0.029250108370068216, 'batch_size': 16, 'dropout_rate': 0.22597922551907568, 'wd': 8.398541678448566e-05, 't_0_loop': 10}. Best is trial 13 with value: 0.5384114583333334.


Epoch 100/100 [Alpha: 0.50] | Train loss: 0.2491 | Val loss: 1.3438 | Val UAR: 0.6328

Fold 1/5
Epoch 1/100 [Alpha: 0.00] | Train loss: 1.7813 | Val loss: 1.6441 | Val UAR: 0.3984
Epoch 10/100 [Alpha: 0.30] | Train loss: 0.1724 | Val loss: 1.5326 | Val UAR: 0.5078
Epoch 20/100 [Alpha: 0.50] | Train loss: 0.1140 | Val loss: 1.9339 | Val UAR: 0.4922
Epoch 30/100 [Alpha: 0.50] | Train loss: 0.0710 | Val loss: 1.8834 | Val UAR: 0.5234
Epoch 40/100 [Alpha: 0.50] | Train loss: 0.0496 | Val loss: 2.2133 | Val UAR: 0.5078
Epoch 50/100 [Alpha: 0.50] | Train loss: 0.0763 | Val loss: 2.6913 | Val UAR: 0.4453
Epoch 60/100 [Alpha: 0.50] | Train loss: 0.0519 | Val loss: 2.5959 | Val UAR: 0.4297
Epoch 70/100 [Alpha: 0.50] | Train loss: 0.0592 | Val loss: 2.3859 | Val UAR: 0.5234
Epoch 80/100 [Alpha: 0.50] | Train loss: 0.0406 | Val loss: 2.5188 | Val UAR: 0.4922
Epoch 90/100 [Alpha: 0.50] | Train loss: 0.0752 | Val loss: 2.4198 | Val UAR: 0.4688
Epoch 100/100 [Alpha: 0.50] | Train loss: 0.0604 | Val 

[I 2026-07-07 16:00:26,052] Trial 31 finished with value: 0.4694010416666667 and parameters: {'lr': 0.011469852949386447, 'batch_size': 16, 'dropout_rate': 0.26962466577098737, 'wd': 2.00903250685809e-05, 't_0_loop': 10}. Best is trial 13 with value: 0.5384114583333334.



Fold 1/5
Epoch 1/100 [Alpha: 0.00] | Train loss: 1.8446 | Val loss: 1.7593 | Val UAR: 0.3125
Epoch 10/100 [Alpha: 0.30] | Train loss: 0.5753 | Val loss: 1.9345 | Val UAR: 0.3906
Epoch 20/100 [Alpha: 0.50] | Train loss: 0.3258 | Val loss: 2.0296 | Val UAR: 0.3906
Epoch 30/100 [Alpha: 0.50] | Train loss: 0.1191 | Val loss: 2.0714 | Val UAR: 0.5000
Epoch 40/100 [Alpha: 0.50] | Train loss: 0.1974 | Val loss: 2.2907 | Val UAR: 0.4062
Epoch 50/100 [Alpha: 0.50] | Train loss: 0.1411 | Val loss: 1.8033 | Val UAR: 0.5312
Epoch 60/100 [Alpha: 0.50] | Train loss: 0.0601 | Val loss: 2.3771 | Val UAR: 0.4609
Epoch 70/100 [Alpha: 0.50] | Train loss: 0.1100 | Val loss: 2.0315 | Val UAR: 0.5469
Epoch 80/100 [Alpha: 0.50] | Train loss: 0.1002 | Val loss: 1.7936 | Val UAR: 0.5156
Epoch 90/100 [Alpha: 0.50] | Train loss: 0.0604 | Val loss: 1.9957 | Val UAR: 0.5312
Epoch 100/100 [Alpha: 0.50] | Train loss: 0.2193 | Val loss: 1.9213 | Val UAR: 0.5391

Fold 2/5
Epoch 1/100 [Alpha: 0.00] | Train loss: 1.802

In [ ]:
import optuna

# Sử dụng optuna.get_all_study_summaries để tự lấy tên study có trong file
summaries = optuna.get_all_study_summaries(storage="sqlite:///optuna_h64_triplet.db")
best_study_name = summaries[0].study_name

# Load study với tên vừa tìm được
study = optuna.load_study(
    study_name=best_study_name,
    storage="sqlite:///optuna_h64_triplet.db"
)

print(f"Đã load thành công study: {best_study_name}")
print("Bộ siêu tham số tốt nhất tìm được:")
print(study.best_params)
print(f"Độ chính xác (UAR) cao nhất đạt được: {study.best_value:.4f}")

Đã load thành công study: ResLSTM_SA_H64_Triplet_tuning
Bộ siêu tham số tốt nhất tìm được:
{'lr': 0.0098287647628273, 'batch_size': 16, 'dropout_rate': 0.19569955017319013, 'wd': 0.0017599300920975779, 't_0_loop': 10}
Độ chính xác (UAR) cao nhất đạt được: 0.5384


## 10-Run Statistical Evaluation

Run 10 independent training cycles with the best hyperparameters to get mean ± std UAR.

In [ ]:
import os
from pytorch_metric_learning import losses, miners

os.makedirs(MODEL_BACKUP_DIR, exist_ok=True)

# Load best params from Optuna
study = optuna.load_study(study_name=STUDY_NAME, storage=SQLITE_STORAGE_URL)
best_params = study.best_params
print("Using hyperparameters:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

results = []
for run in range(10):
    print(f"\n{'='*60}")
    print(f"Run {run + 1}/10")
    print(f"{'='*60}")

    set_seed(42 + run * 100)

    def model_factory():
        return ResLSTM_SA(
            input_size=INPUT_SIZE, hidden_size=HIDDEN_SIZE,
            num_layers=NUM_LAYERS, num_classes=NUM_CLASSES,
            dropout_p=best_params['dropout_rate'], device=str(DEVICE)
        ).to(DEVICE)

    def optimizer_factory(model):
        return optim.Adam(model.parameters(), lr=best_params['lr'],
                          weight_decay=best_params['wd'])

    def scheduler_factory(optimizer):
        return optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=best_params['t_0_loop'], T_mult=1,
            eta_min=best_params['lr'] * 0.01)

    loss_fn_ce = nn.CrossEntropyLoss()
    loss_fn_triplet = losses.TripletMarginLoss(margin=0.3)
    miner = miners.MultiSimilarityMiner()
    target_alpha = 0.5 

    model_name = f"ResLSTM_SA_H{HIDDEN_SIZE}"
    init_model_path = os.path.join(MODEL_BACKUP_DIR, f"init_{model_name}_run{run}.pt")
    model_path = os.path.join(MODEL_BACKUP_DIR, f"best_{model_name}_run{run}.pt")

    oof_preds, oof_targets = k_fold_cv(
        dataset=dataset, model_factory=model_factory, 
        loss_fn_ce=loss_fn_ce, loss_fn_triplet=loss_fn_triplet, miner=miner, alpha=target_alpha,
        optimizer_factory=optimizer_factory, lr_scheduler_factory=scheduler_factory,
        n_epochs=N_EPOCHS, k_fold=K_FOLD, batch_size=best_params['batch_size'],
        init_model_path=init_model_path, model_path=model_path)

    all_preds = torch.cat([p for p in oof_preds])
    all_targets = torch.cat([t for t in oof_targets])

    evaluator = get_default_evaluator()
    macro_metric = Recall(average=True)
    macro_metric.attach(evaluator, "macro_recall")
    evaluator.terminate()
    state = evaluator.run([[all_preds, all_targets]])
    uar = state.metrics["macro_recall"]

    _m = model_factory()
    num_params = sum(p.numel() for p in _m.parameters() if p.requires_grad)

    results.append({"run": run + 1, "uar": uar, "num_params": num_params})
    print(f"Run {run + 1}: UAR = {uar:.4f}, Params = {num_params}")

# Statistics
uars = [r["uar"] for r in results]
mean_uar = np.mean(uars)
std_uar = np.std(uars)
print(f"\nFinal: UAR = {mean_uar:.4f} +/- {std_uar:.4f}")
print(f"Num params: {results[0]['num_params']}")

Using hyperparameters:
  lr: 0.024788433285349035
  batch_size: 16
  dropout_rate: 0.08697636034984436
  wd: 0.004634460388759558
  t_0_loop: 5

Run 1/10

Fold 1/5


C:\Users\duong\AppData\Local\Temp\ipykernel_5652\3143617173.py:10: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None
C:\Users\duong\AppData\Local\Temp\ipykernel_5652\3143617173.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 1/100 | Train loss: 1.8601 | Val loss: 1.7310 | Val UAR: 0.3672
Epoch 10/100 | Train loss: 0.4543 | Val loss: 1.2542 | Val UAR: 0.5391
Epoch 20/100 | Train loss: 0.3953 | Val loss: 1.4554 | Val UAR: 0.5391
Epoch 30/100 | Train loss: 0.3731 | Val loss: 1.2951 | Val UAR: 0.5703
Epoch 40/100 | Train loss: 0.4194 | Val loss: 1.3279 | Val UAR: 0.4766
Epoch 50/100 | Train loss: 0.3579 | Val loss: 1.5525 | Val UAR: 0.4219
Epoch 60/100 | Train loss: 0.3585 | Val loss: 1.5374 | Val UAR: 0.4844
Epoch 70/100 | Train loss: 0.3262 | Val loss: 1.4122 | Val UAR: 0.5078
Epoch 80/100 | Train loss: 0.3231 | Val loss: 1.5418 | Val UAR: 0.5000
Epoch 90/100 | Train loss: 0.3513 | Val loss: 1.4997 | Val UAR: 0.5312
Epoch 100/100 | Train loss: 0.3349 | Val loss: 1.7732 | Val UAR: 0.4688


C:\Users\duong\AppData\Local\Temp\ipykernel_5652\3143617173.py:75: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path))



Fold 2/5
Epoch 1/100 | Train loss: 1.8339 | Val loss: 1.9092 | Val UAR: 0.2812
Epoch 10/100 | Train loss: 0.4422 | Val loss: 1.5792 | Val UAR: 0.4609
Epoch 20/100 | Train loss: 0.3780 | Val loss: 1.4948 | Val UAR: 0.5000
Epoch 30/100 | Train loss: 0.3543 | Val loss: 1.4885 | Val UAR: 0.5391
Epoch 40/100 | Train loss: 0.3532 | Val loss: 1.6099 | Val UAR: 0.4609
Epoch 50/100 | Train loss: 0.3807 | Val loss: 1.5284 | Val UAR: 0.4844
Epoch 60/100 | Train loss: 0.3502 | Val loss: 1.7980 | Val UAR: 0.4609
Epoch 70/100 | Train loss: 0.3830 | Val loss: 1.7918 | Val UAR: 0.4922
Epoch 80/100 | Train loss: 0.3700 | Val loss: 1.6341 | Val UAR: 0.5234
Epoch 90/100 | Train loss: 0.3426 | Val loss: 1.7756 | Val UAR: 0.4766
Epoch 100/100 | Train loss: 0.3304 | Val loss: 1.5165 | Val UAR: 0.5391

Fold 3/5
Epoch 1/100 | Train loss: 1.8416 | Val loss: 2.0837 | Val UAR: 0.3047
Epoch 10/100 | Train loss: 0.4081 | Val loss: 1.4494 | Val UAR: 0.5312
Epoch 20/100 | Train loss: 0.3653 | Val loss: 1.5449 | Val

## Summary

This notebook reproduces the full experimental pipeline from the DSPA 2026 paper:

1. **Feature Extraction**: 34 MFCC + 12 chroma = 46-dim features, two-pass global normalization
2. **Model**: ResLSTM-SA — residual LSTM with soft attention
3. **Hyperparameter Tuning**: Bayesian optimization with Optuna (TPE sampler)
4. **Evaluation**: 10 independent runs, speaker-independent 5-fold CV, reporting UAR mean ± std

### Key Results (from paper)

| Model | Hidden Size | Params | UAR (mean ± std) | Max UAR |
|-------|-------------|--------|------------------|---------|
| ResLSTM-SA-h64 | 64 | 46.8k | 0.6232 ± 0.0119 | **0.6517** |
| ResLSTM-SA-h128 | 128 | 108.9k | 0.6107 ± 0.0134 | 0.6348 |
| ResLSTM-SA-h32 | 32 | 28.0k | 0.6130 ± 0.0111 | 0.6315 |

### References

- Krasnoproshin, D. & Vashkevich, M. (2026). *Speech Emotion Recognition using Attention-based LSTM-Network with Residual Connection.* DSPA 2026.
- Mirsamadi, S., Barsoum, E., & Zhang, C. (2017). *Automatic speech emotion recognition using recurrent neural networks with local attention.* ICASSP 2017.
- Livingstone, S. R. & Russo, F. A. (2018). *The RAVDESS database.* PLoS ONE.
- Akiba, T. et al. (2019). *Optuna: A Next-generation Hyperparameter Optimization Framework.* KDD 2019.